# GPT: a hands-on implementation

**Track:** Language models · **Guide:** [`035-gpt.md`](../guides/concepts/035-gpt.md)

GPT-style models are decoder-only Transformers trained to predict the next token, then adapted for useful generation.

> **Intuition:** It writes one word at a time, using everything already written as context.

This lesson builds a small working example in stages: prepare inputs, implement
the core idea, run it on a realistic toy case, and inspect its output and limits.
It runs offline unless a cell is explicitly marked as an optional provider call.

```mermaid
flowchart LR
    A[Prepare inputs] --> B[Implement GPT]
    B --> C[Run on example]
    C --> D[Evaluate and inspect]
```

**Case study:** Generate a draft answer, then require retrieval, validation, and human review where correctness matters.

Run the cells from top to bottom. To install the notebook environment, follow
[`guides/SETUP.md`](../guides/SETUP.md). The toy data and implementation are for
learning, not a production quality, safety, or security guarantee.


## 1. Build a next-token training corpus

GPT-style decoder-only models learn to predict the next token from the prefix. This miniature uses a token bigram model.


In [ ]:
from collections import Counter, defaultdict
from itertools import pairwise

corpus = [
    "small models learn patterns",
    "large models learn language patterns",
    "language models predict the next token",
    "models predict one token at a time",
]
transitions = defaultdict(Counter)
for sentence in corpus:
    words = ["<BOS>", *sentence.split(), "<EOS>"]
    for current, following in pairwise(words):
        transitions[current][following] += 1
print("transitions from 'models':", transitions["models"])

## 2. Implement next-token prediction

Choose the most frequent successor for a deterministic demonstration. A trained GPT uses learned neural-network scores and sampling.


In [ ]:
def next_token(prefix):
    current = prefix.split()[-1] if prefix.split() else "<BOS>"
    choices = transitions[current]
    if not choices:
        return "<EOS>"
    return choices.most_common(1)[0][0]

print("after 'language':", next_token("language"))
print("after 'models':", next_token("models"))

## 3. Generate autoregressively

Feed each generated token back as context, stopping at an end token or the maximum length.


In [ ]:
def generate(prompt, max_tokens=8):
    generated = prompt.split()
    for _ in range(max_tokens):
        token = next_token(" ".join(generated))
        if token == "<EOS>":
            break
        generated.append(token)
    return " ".join(generated)

print(generate("language models"))

## Wrap-up and next steps

- Identify what was learned, stored, retrieved, or generated in this implementation.
- Change an input and observe how the output changes.
- Test one failure case and decide when the system should abstain or ask for help.
- Replace toy data/components only after you can evaluate the change against a
  held-out or representative test set.

Continue with the [concept guide](../guides/concepts/035-gpt.md) and the
[learning roadmap](../guides/ROADMAP.md).
